In [ ]:
import pandas as pd
import numpy as np
import os
import sys

import logging
from _utilities.stupidloggen import StupidLogGenerator

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.metrics.pairwise import euclidean_distances

ROOT_DIR = os.environ["VIRTUAL_ENV"][:-5]
sys.path.append(ROOT_DIR)

In [ ]:
PATH_REAL_DATA = "data/preprocessed_real_data.csv"
PATH_SYNTHETIC_DATA = "data/synthetic/"
PATH_FOLDER_MODELS = "models/"

In [ ]:
logger = StupidLogGenerator.initLogger()

In [ ]:
# Real Dataset
real_df = pd.read_csv(PATH_REAL_DATA)

# Target Column
target_feature = "is_recid"

# Sensitive Column
sensitive_feature = "sex"

# Synthetic Datasets
synthetic_datasets = {}

# Speciy column types and order
X_real = real_df.drop(columns=[target_feature])
CATEGORICAL_COLS = X_real.select_dtypes(include=["object"]).columns.tolist()
NUMERICAL_COLS = X_real.select_dtypes(exclude=["object"]).columns.tolist()
FEATURE_ORDER = X_real.columns.tolist()  # The exact order of columns expected

In [ ]:
# Load synthetic datasets
dir_list = os.listdir(PATH_SYNTHETIC_DATA)

for file_name in dir_list:
    model_name = file_name.replace("_synthetic.csv", "").replace("lambda_", "")
    synthetic_datasets[model_name] = pd.read_csv(PATH_SYNTHETIC_DATA + file_name)

## Downstream Classifier

### Preprocessing

In [ ]:
real_train_df, real_test_df = train_test_split(real_df, test_size=0.2, random_state=42)


def build_pipeline(categorical_cols: list[str], numerical_cols: list[str]) -> Pipeline:
    """Builds a preprocessing + Logistic Regression pipeline"""

    preprocessor = ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
            ("num", "passthrough", numerical_cols),
        ]
    )

    pipeline = Pipeline(
        [
            ("preprocessor", preprocessor),
            ("classifier", LogisticRegression(max_iter=1000)),
        ]
    )

    return pipeline

### Train the classifier

In [ ]:
def train_pipeline(
    train_df: pd.DataFrame,
    target_col: str,
    categorical_cols: list[str],
    numeric_cols: list[str],
    feature_order: list[str],
) -> Pipeline:
    """Trains a pipeline on a given dataset (real or synthetic)."""

    X_train = train_df.drop(columns=[target_col])
    y_train = train_df[target_col]

    X_train = X_train.reindex(columns=feature_order)

    pipeline = build_pipeline(categorical_cols, numeric_cols)
    pipeline.fit(X_train, y_train)

    return pipeline

### Run the training on the real dataset and the synthetic dataset 

In [ ]:
# Train on the real dataset
real_model = train_pipeline(
    real_train_df, target_feature, CATEGORICAL_COLS, NUMERICAL_COLS, FEATURE_ORDER
)

synthetic_models = {}
one_class_datasets = []

# Train synthetic models
for model_name, df in synthetic_datasets.items():
    n_unique = df[target_feature].nunique()

    if n_unique < 2:
        logger.warning(f"Skipping {model_name}: Found only {n_unique} class")

    try:
        synthetic_models[model_name] = train_pipeline(
            df, target_feature, CATEGORICAL_COLS, NUMERICAL_COLS, FEATURE_ORDER
        )
        logger.info(f"Successfully trained: {model_name}")
    except Exception as e:
        logger.error(f"Failed to train {model_name}: {e}")

## Evaluation Definitions

#### Fairness Metrics

In [ ]:
# Fairness Metrics Definition
# Note to self: Always sort groups first since order isn't guaranteed i.e. results may flip


def demographic_parity(y_pred: pd.Series, sensitive: pd.Series) -> float:
    groups = sorted(sensitive.unique())
    rates = [y_pred[sensitive == g].mean() for g in groups]
    return abs(rates[0] - rates[1])


def disparate_impact(y_pred: pd.Series, sensitive: pd.Series) -> float:
    groups = sorted(sensitive.unique())
    rate_0 = y_pred[sensitive == groups[0]].mean()
    rate_1 = y_pred[sensitive == groups[1]].mean()
    return min(rate_0 / (rate_1 + 1e-6), rate_1 / (rate_0 + 1e-6))


def equal_opportunity(y_true: pd.Series, y_pred: pd.Series, sensitive: pd.Series) -> float:
    groups = sorted(sensitive.unique())
    tprs = []

    for g in groups:
        mask = sensitive == g
        tp = ((y_pred == 1) & (y_true == 1) & mask).sum()
        fn = ((y_pred == 0) & (y_true == 1) & mask).sum()
        tprs.append(tp / (tp + fn + 1e-6))

    return abs(tprs[0] - tprs[1])


def equalized_odds(y_true: pd.Series, y_pred: pd.Series, sensitive: pd.Series) -> float:
    groups = sorted(sensitive.unique())
    tpr_list, fpr_list = [], []

    for g in groups:
        mask = sensitive == g

        tp = ((y_pred == 1) & (y_true == 1) & mask).sum()
        fn = ((y_pred == 0) & (y_true == 1) & mask).sum()
        fp = ((y_pred == 1) & (y_true == 0) & mask).sum()
        tn = ((y_pred == 0) & (y_true == 0) & mask).sum()

        tpr = tp / (tp + fn + 1e-6)
        fpr = fp / (fp + tn + 1e-6)

        tpr_list.append(tpr)
        fpr_list.append(fpr)

    return abs(tpr_list[0] - tpr_list[1]) + abs(fpr_list[0] - fpr_list[1])


# Fairness Metrics Evaluation
def evaluate_fairness(
    model: Pipeline, test_df: pd.DataFrame, target_col: str, sensitive_col: str
) -> dict:
    """
    Evaluates fairness metrics on the real test set.
    """

    # Split test data
    X_test = test_df.drop(columns=[target_col])
    y_true = test_df[target_col]
    sensitive = test_df[sensitive_col]

    # Predict using trained pipeline
    y_pred = model.predict(X_test)

    # Compute fairness metrics
    results = {
        "demographic_parity": demographic_parity(y_pred, sensitive),
        "disparate_impact": disparate_impact(y_pred, sensitive),
        "equal_opportunity": equal_opportunity(y_true, y_pred, sensitive),
        "equalized_odds": equalized_odds(y_true, y_pred, sensitive),
    }

    return results

In [ ]:
# Comparison of Fairness Across Datasets
fairness_results = {}

# Real dataset baseline
fairness_results["Real"] = evaluate_fairness(
    real_model,
    real_test_df,
    target_col=target_feature,
    sensitive_col=sensitive_feature,
)

# Synthetic models
for name, model in synthetic_models.items():
    fairness_results[name] = evaluate_fairness(
        model, real_test_df, target_col=target_feature, sensitive_col=sensitive_feature
    )

fairness_df = pd.DataFrame(fairness_results).T
fairness_df

#### Synthetic Data Realism Metric

In [ ]:
def estimate_gamma(X: np.ndarray) -> float:
    """
    Estimates gamma using the median heuristic.
    """
    dists = euclidean_distances(X, X)
    dists = dists[dists > 0]  # remove zeros (distance to itself)

    median_dist = np.median(dists)

    gamma = 1 / (2 * (median_dist**2 + 1e-6))
    return gamma


# Preprocessing for MMD
def preprocess_mmd(df: pd.DataFrame, target_col: str) -> pd.DataFrame:
    """
    Converts dataframe into numeric matrix for MMD:
    - Drops target column
    - One-hot encodes categorical features
    """
    X = df.drop(columns=[target_col])
    X = pd.get_dummies(X, drop_first=True)
    return X


# MMD Computation
def compute_mmd(X_real: np.ndarray, X_synth: np.ndarray, gamma=1.0) -> float:
    """
    Computes Maximum Mean Discrepancy using RBF kernel.
    """
    K_xx = rbf_kernel(X_real, X_real, gamma=gamma)
    K_yy = rbf_kernel(X_synth, X_synth, gamma=gamma)
    K_xy = rbf_kernel(X_real, X_synth, gamma=gamma)

    mmd = K_xx.mean() + K_yy.mean() - 2 * K_xy.mean()

    return mmd


# Main Evaluation Function
def evaluate_realism(
    real_df: pd.DataFrame, synth_df: pd.DataFrame, target_col: str, sample_size=1000
) -> float:
    """
    Computes MMD using random sampling + median heuristic for gamma.
    """
    X_real = preprocess_mmd(real_df, target_col)
    X_synth = preprocess_mmd(synth_df, target_col)

    X_synth = X_synth.reindex(columns=X_real.columns, fill_value=0)

    # --- SAMPLE ---
    X_real_sample = X_real.sample(n=min(sample_size, len(X_real)), random_state=42)
    X_synth_sample = X_synth.sample(n=min(sample_size, len(X_synth)), random_state=42)

    # --- Estimate gamma from REAL data ---
    gamma = estimate_gamma(X_real_sample.values)

    return compute_mmd(
        X_real_sample.values,
        X_synth_sample.values,
        gamma=gamma,
    )

In [ ]:
# Run for All Synthetic Datasets
realism_results = {}

for model_name, synth_df in synthetic_datasets.items():
    realism_results[model_name] = evaluate_realism(
        real_train_df, synth_df, target_col=target_feature, sample_size=1000
    )

realism_df = pd.DataFrame.from_dict(realism_results, orient="index", columns=["MMD"])

realism_df

#### Utility

In [ ]:
def evaluate_utility(model: Pipeline, test_df: pd.DataFrame, target_col: str) -> dict:
    """
    Evaluates predictive performance using TSTR:
    Train on (real/synthetic) → Test on REAL test data
    """
    X_test = test_df.drop(columns=[target_col])
    y_true = test_df[target_col]
    y_pred = model.predict(X_test)

    # Probabilities
    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_test)[:, 1]
    else:
        y_prob = y_pred

    results = {
        "accuracy": accuracy_score(y_true, y_pred),
        "auc": roc_auc_score(y_true, y_prob),
    }

    return results

In [ ]:
utility_results = {}

# Real dataset baseline
utility_results["Real"] = evaluate_utility(real_model, real_test_df, target_col=target_feature)

# Synthetic models
for model_name, model in synthetic_models.items():
    utility_results[model_name] = evaluate_utility(
        model, real_test_df, target_col=target_feature
    )

utility_df = pd.DataFrame(utility_results).T
utility_df

## Evaluation Comparison

In [ ]:
logger.setLevel(logging.INFO)

In [ ]:
final_df = pd.concat([utility_df, fairness_df, realism_df], axis=1)
final_df = final_df.fillna("-")

final_df = final_df[
    [
        "accuracy",
        "auc",
        "demographic_parity",
        "disparate_impact",
        "equal_opportunity",
        "equalized_odds",
        "MMD",
    ]
]

final_df